# Bottleneck Analysis: Wild-Type vs Gene Deletion Mutants

This notebook demonstrates a basic FBA workflow using the *E. coli* core model:
1. Load the model and configure the solver
2. Run FBA on the wild-type
3. Identify non-essential genes via single gene deletion screening
4. Delete selected non-essential genes
5. Compare metabolic bottlenecks (FVA, shadow prices) between WT and mutants

## 1. Load model and configure solver

In [ ]:
import cobra
from cobra.flux_analysis import (
    flux_variability_analysis,
    single_gene_deletion,
    pfba,
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the E. coli core model (72 metabolites, 95 reactions, 137 genes)
model = cobra.io.load_model("textbook")
model.solver = "highs"

print(f"Model: {model.id}")
print(f"Reactions: {len(model.reactions)}")
print(f"Metabolites: {len(model.metabolites)}")
print(f"Genes: {len(model.genes)}")
print(f"Solver: {model.solver.interface.__name__}")

## 2. Wild-type FBA

In [ ]:
wt_solution = pfba(model)
print(f"WT growth rate: {wt_solution.objective_value:.4f} h⁻¹")
print(f"WT flux through biomass: {wt_solution.fluxes['BIOMASS_Ecoli_core_w_GAM']:.4f}")
wt_solution.fluxes[wt_solution.fluxes.abs() > 1e-6].sort_values().head(10)

## 3. Identify non-essential genes

Screen all single gene deletions to find genes whose removal still permits growth.

In [ ]:
deletion_results = single_gene_deletion(model)

# Classify genes by essentiality (< 1% of WT growth = essential)
wt_growth = wt_solution.objective_value
deletion_results["growth_fraction"] = deletion_results["growth"] / wt_growth
deletion_results["gene"] = deletion_results["ids"].apply(lambda x: list(x)[0])

essential = deletion_results[deletion_results["growth_fraction"] < 0.01]
non_essential = deletion_results[deletion_results["growth_fraction"] >= 0.01].sort_values("growth_fraction")

print(f"Essential genes: {len(essential)}")
print(f"Non-essential genes: {len(non_essential)}")
print(f"\nNon-essential genes with reduced growth (partial effect):")
partial = non_essential[non_essential["growth_fraction"] < 0.99]
partial[["gene", "growth", "growth_fraction"]].sort_values("growth_fraction")

## 4. Select and delete non-essential genes

We pick genes that are non-essential but have a measurable impact on flux distribution,
making bottleneck shifts visible. We'll delete them individually and as a combined mutant.

In [ ]:
# Pick non-essential genes with partial growth reduction
# If no partial-effect genes exist, pick from fully non-essential genes
if len(partial) >= 3:
    genes_to_delete = partial["gene"].head(3).tolist()
else:
    genes_to_delete = non_essential["gene"].head(3).tolist()

print("Genes selected for deletion:")
for g in genes_to_delete:
    gene_obj = model.genes.get_by_id(g)
    rxns = [r.id for r in gene_obj.reactions]
    growth_frac = deletion_results.loc[
        deletion_results["gene"] == g, "growth_fraction"
    ].values[0]
    print(f"  {g}: reactions={rxns}, growth fraction={growth_frac:.3f}")

In [ ]:
# Create mutant model with all selected genes knocked out
mutant = model.copy()
for gene_id in genes_to_delete:
    gene = mutant.genes.get_by_id(gene_id)
    gene.knock_out()

mutant_solution = pfba(mutant)
print(f"Mutant growth rate: {mutant_solution.objective_value:.4f} h⁻¹")
print(f"Growth reduction: {(1 - mutant_solution.objective_value / wt_growth) * 100:.1f}%")

## 5. Bottleneck analysis: FVA comparison

Flux Variability Analysis reveals how much flexibility each reaction has. Reactions with
narrow FVA ranges are tightly constrained — these are metabolic bottlenecks. Comparing
WT and mutant FVA shows which bottlenecks shift after gene deletion.

In [ ]:
# Run FVA at 90% of optimal growth for both WT and mutant
fva_wt = flux_variability_analysis(model, fraction_of_optimum=0.9)
fva_mutant = flux_variability_analysis(mutant, fraction_of_optimum=0.9)

# Compute flux ranges
fva_wt["range"] = fva_wt["maximum"] - fva_wt["minimum"]
fva_mutant["range"] = fva_mutant["maximum"] - fva_mutant["minimum"]

# Combine into a comparison table
fva_compare = pd.DataFrame({
    "wt_min": fva_wt["minimum"],
    "wt_max": fva_wt["maximum"],
    "wt_range": fva_wt["range"],
    "mut_min": fva_mutant["minimum"],
    "mut_max": fva_mutant["maximum"],
    "mut_range": fva_mutant["range"],
})
fva_compare["range_change"] = fva_compare["mut_range"] - fva_compare["wt_range"]

print("Reactions that became MORE constrained in the mutant (new bottlenecks):")
tighter = fva_compare[fva_compare["range_change"] < -1e-3].sort_values("range_change")
tighter.head(15)

In [ ]:
print("Reactions that became LESS constrained in the mutant (released bottlenecks):")
looser = fva_compare[fva_compare["range_change"] > 1e-3].sort_values("range_change", ascending=False)
looser.head(15)

In [ ]:
# Visualize the top reactions with changed flux ranges
top_changed = fva_compare.reindex(
    fva_compare["range_change"].abs().nlargest(20).index
)

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(top_changed))
ax.barh(x, top_changed["wt_range"], height=0.4, label="WT", color="steelblue", align="center")
ax.barh([i + 0.4 for i in x], top_changed["mut_range"], height=0.4, label="Mutant", color="coral", align="center")
ax.set_yticks([i + 0.2 for i in x])
ax.set_yticklabels(top_changed.index, fontsize=9)
ax.set_xlabel("Flux range (FVA)")
ax.set_title("FVA Range Comparison: WT vs Mutant (top 20 changed reactions)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Shadow price analysis

Shadow prices indicate how much the objective (growth) would improve if a metabolite's
availability increased by one unit. Large absolute shadow prices identify limiting
metabolites — the biochemical bottlenecks.

In [ ]:
# Shadow prices from the WT and mutant FBA solutions
shadow_wt = wt_solution.shadow_prices
shadow_mut = mutant_solution.shadow_prices

shadow_compare = pd.DataFrame({
    "wt_shadow": shadow_wt,
    "mut_shadow": shadow_mut,
})
shadow_compare["change"] = shadow_compare["mut_shadow"] - shadow_compare["wt_shadow"]

# Show metabolites with the largest shadow price changes
print("Metabolites with largest shadow price changes (new/shifted bottlenecks):")
shadow_compare.reindex(
    shadow_compare["change"].abs().nlargest(15).index
)[["wt_shadow", "mut_shadow", "change"]]

In [ ]:
# Visualize top shadow price changes
top_shadow = shadow_compare.reindex(
    shadow_compare["change"].abs().nlargest(15).index
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute shadow prices
ax = axes[0]
top_shadow[["wt_shadow", "mut_shadow"]].plot.barh(ax=ax, color=["steelblue", "coral"])
ax.set_xlabel("Shadow price")
ax.set_title("Shadow Prices: WT vs Mutant")
ax.legend(["WT", "Mutant"])

# Change in shadow prices
ax = axes[1]
colors = ["green" if v > 0 else "red" for v in top_shadow["change"]]
top_shadow["change"].plot.barh(ax=ax, color=colors)
ax.set_xlabel("Shadow price change (mutant − WT)")
ax.set_title("Shift in Metabolite Limitation")

plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
print("=" * 60)
print("BOTTLENECK ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nModel: {model.id}")
print(f"Genes deleted: {genes_to_delete}")
print(f"\nGrowth rates:")
print(f"  WT:     {wt_growth:.4f} h⁻¹")
print(f"  Mutant: {mutant_solution.objective_value:.4f} h⁻¹")
print(f"  Change: {(mutant_solution.objective_value / wt_growth - 1) * 100:+.1f}%")
print(f"\nFVA bottleneck shifts (reactions):")
print(f"  Became more constrained: {len(tighter)}")
print(f"  Became less constrained: {len(looser)}")
print(f"\nTop new bottleneck reactions (tighter FVA range):")
for rxn_id in tighter.head(5).index:
    rxn = model.reactions.get_by_id(rxn_id)
    print(f"  {rxn_id}: {rxn.name}")
print(f"\nTop limiting metabolites shifted (by shadow price change):")
for met_id in top_shadow.head(5).index:
    met = model.metabolites.get_by_id(met_id)
    print(f"  {met_id}: {met.name}")